In [13]:
import pandas as pd
df_train=pd.read_csv('data/train.csv')
df_test=pd.read_csv('data/test.csv')
df_train_merged = pd.read_csv('data/train_merged.csv')

In [14]:
df_train_merged.head()

,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,SWI,MEDIUM,Mexico City Grand Prix,2023,0,6,1,6.0,12,83.921,-21.244,-10.320,0.084507,0.0,0.0
1,TRU,HARD,Italian Grand Prix,2024,0,24,2,17.0,15,83.845,-22.913,-33.696,0.311688,-9.0,1.0
2,TSU,MEDIUM,Monaco Grand Prix,2023,0,23,1,23.0,9,79.239,0.087,-12.078,0.302632,0.0,0.0
3,PEA,HARD,Italian Grand Prix,2022,1,50,2,33.0,11,87.076,-13.929,-31.804,0.694444,3.0,1.0
4,ANT,HARD,Monaco Grand Prix,2025,0,49,1,49.0,12,78.328,-0.516,-33.315,0.653333,0.0,0.0


In [15]:
len(df_train_merged)

540445

In [16]:
config_json = """
{
  "target_column": "PitNextLap",
  "global_random_state": 42,
  "files": [
    {
      "file_id": 1,
      "chunk_size": 100000,
      "sample_size": 10000,
      "ratio_1": 0.20
    },
    {
      "file_id": 2,
      "chunk_size": 100000,
      "sample_size": 10000,
      "ratio_1": 0.15
    },
    {
      "file_id": 3,
      "chunk_size": 100000,
      "sample_size": 10000,
      "ratio_1": 0.25
    },
    {
      "file_id": 4,
      "chunk_size": 100000,
      "sample_size": 10000,
      "ratio_1": 0.30
    },
    {
      "file_id": 5,
      "chunk_size": 100000,
      "sample_size": 10000,
      "ratio_1": 0.10
    }
  ]
}
"""


In [19]:
import json
import numpy as np
import pandas as pd

# ==========================================================
# 1. JSON CONFIGURATION (Edit distributions per file here)
# ==========================================================


# ==========================================================
# 2. SAMPLING FUNCTION
# ==========================================================
def sample_with_custom_ratio(
    df, target_col, sample_size, ratio_1, random_state=42
):
    """Extracts a stratified sample based on the specific file's ratio_1 config."""
    n_1 = int(sample_size * ratio_1)
    n_0 = sample_size - n_1

    df_1 = df[df[target_col] == 1]
    df_0 = df[df[target_col] == 0]

    if len(df_1) < n_1 or len(df_0) < n_0:
        raise ValueError(
            f"Insufficient data in chunk to fulfill target ratios. "
            f"Required: {n_1} ones (found {len(df_1)}), {n_0} zeros (found {len(df_0)})."
        )

    sample_1 = df_1.sample(n=n_1, random_state=random_state)
    sample_0 = df_0.sample(n=n_0, random_state=random_state)

    sampled_df = (
        pd.concat([sample_1, sample_0])
        .sample(frac=1, random_state=random_state)
        .reset_index(drop=True)
    )
    return sampled_df


# ==========================================================
# 3. MAIN EXECUTION PIPELINE
# ==========================================================
# Parse JSON config (or load from file: config = json.load(open('config.json')))
config = json.loads(config_json)
target_col = config["target_column"]
seed = config["global_random_state"]

# Load your 500k dataset (Replace with pd.read_csv("your_dataset.csv"))

df= df_train_merged.copy()
# Shuffle dataset once globally
df = df.sample(frac=1, random_state=seed).reset_index(drop=True)

# Iterate through each file configuration in the JSON
start_idx = 0
for file_cfg in config["files"]:
    file_id = file_cfg["file_id"]
    chunk_size = file_cfg["chunk_size"]
    sample_size = file_cfg["sample_size"]
    ratio_1 = file_cfg["ratio_1"]

    end_idx = start_idx + chunk_size

    # 1. Extract chunk
    chunk_df = df.iloc[start_idx:end_idx].copy().reset_index(drop=True)

    # 2. Draw target sample using file-specific ratio
    sampled_df = sample_with_custom_ratio(
        df=chunk_df,
        target_col=target_col,
        sample_size=sample_size,
        ratio_1=ratio_1,
        random_state=seed + file_id,
    )

    
   

    # Output verification
    counts = sampled_df[target_col].value_counts()
    percentages = sampled_df[target_col].value_counts(normalize=True) * 100

    sampled_df.rename(columns={target_col: 'ground_truth'}, inplace=True)
    sampled_df['ground_truth'] = sampled_df['ground_truth'].astype(int)
     # 3. Save outputs (Uncomment to write CSV files)
    # chunk_df.to_csv(f"dataset_part_{file_id}_{chunk_size}k.csv", index=False)
    sampled_df.to_csv(f"Sampling_data_to_test/sample_part_{file_id}_{sample_size}k.csv", index=False)

    print(f"=== File {file_id} Configuration ===")
    print(
        f"Target Ratio Config : {ratio_1 * 100:.0f}% '1's / {(1 - ratio_1) * 100:.0f}% '0's"
    )
    print(f"Sample Records Count:\n{counts.to_string()}")
    print(f"Actual Ratio (%):\n{percentages.to_string()}\n")

    # Shift pointer for next dataset chunk
    start_idx = end_idx

=== File 1 Configuration ===
Target Ratio Config : 20% '1's / 80% '0's
Sample Records Count:
PitNextLap
0.0    8000
1.0    2000
Actual Ratio (%):
PitNextLap
0.0    80.0
1.0    20.0

=== File 2 Configuration ===
Target Ratio Config : 15% '1's / 85% '0's
Sample Records Count:
PitNextLap
0.0    8500
1.0    1500
Actual Ratio (%):
PitNextLap
0.0    85.0
1.0    15.0

=== File 3 Configuration ===
Target Ratio Config : 25% '1's / 75% '0's
Sample Records Count:
PitNextLap
0.0    7500
1.0    2500
Actual Ratio (%):
PitNextLap
0.0    75.0
1.0    25.0

=== File 4 Configuration ===
Target Ratio Config : 30% '1's / 70% '0's
Sample Records Count:
PitNextLap
0.0    7000
1.0    3000
Actual Ratio (%):
PitNextLap
0.0    70.0
1.0    30.0

=== File 5 Configuration ===
Target Ratio Config : 10% '1's / 90% '0's
Sample Records Count:
PitNextLap
0.0    9000
1.0    1000
Actual Ratio (%):
PitNextLap
0.0    90.0
1.0    10.0



In [20]:
sampled_df.head()

,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,ground_truth
0,PAL,SOFT,Hungarian Grand Prix,2022,0,64,4,22.0,5,83.538,13.849,-94.906,0.888889,0.0,0
1,ZHO,HARD,Hungarian Grand Prix,2023,0,20,2,15.0,15,85.263,-0.280,-24.675,0.289855,0.0,0
2,D014,MEDIUM,Singapore Grand Prix,2024,0,3,1,3.0,3,99.087,-10.698,-43.728,0.038462,2.0,0
3,LEC,MEDIUM,Dutch Grand Prix,2024,0,9,1,11.0,5,77.837,-33.403,-84.936,0.115385,7.0,0
4,D287,MEDIUM,Miami Grand Prix,2023,0,15,1,15.0,1,92.946,0.033,-9.937,0.263158,0.0,0
